In [1]:
import numpy as np

import matplotlib.pyplot as plt
import pandas as pd

import os
import pickle

from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import pdist
from scipy.stats import zscore
from scipy.cluster.hierarchy import fcluster


In [2]:
from nmf_utils import bandpass_filter_traces, plot_peak_heatmap

In [3]:
def main(path_save, file_stem):
    """
    Analyze peak shapes per ROI, classify fast vs slow events, and compute t2p statistics.

    This function:
    1. Loads traces and peak timing data from previous analysis steps.
    2. Applies bandpass filtering to traces using stored preprocessing parameters.
    3. Extracts peak-aligned segments for each ROI.
    4. Normalizes peak shapes and performs hierarchical clustering.
    5. Classifies peaks into "fast" and "slow" based on time-to-peak (t2p).
    6. Generates and saves heatmaps of clustered peak shapes for each ROI.
    7. Computes summary metrics per ROI:
       - Mean time-to-peak for fast events
       - Mean time-to-peak for slow events
       - Ratio of fast to slow events

    Parameters
    ----------
    path_save : str
        Directory where input data is stored and output figures will be saved.
    file_stem : str
        Base filename used to locate input files and name output files.

    Returns
    -------
    t2p_fast : list of float
        Mean time-to-peak (in seconds) for fast events per ROI.
    t2p_slow : list of float
        Mean time-to-peak (in seconds) for slow events per ROI.
    fast_slow_ratio : list of float
        Ratio of number of fast events to slow events per ROI.

    Notes
    -----
    - Peak segments are extracted starting from peak onset (peaks_start) and extending
      for a fixed number of frames.
    - Clustering is done per ROI using Ward linkage on normalized peak shapes.
    - Fast vs slow clusters are identified based on average time-to-peak within clusters.
    - Additional classification is applied using a fixed frame threshold (thresh).
    - Heatmaps visualize variability of peak shapes and cluster structure per ROI.
    - Results are returned per ROI without saving aggregated statistics.
    """
    #Parameters
    fr = 10  # number of frames extracted for each peak (length of peak segment)
    thresh = 5  # threshold (in frames) to separate fast vs slow peaks

    # Folder to save per-ROI peak heatmaps
    folder_save = os.path.join(path_save, file_stem + "Peaks_heatmaps") # folder to save per-roi peaks heatplots
    os.makedirs(folder_save, exist_ok=True)

    #load traces
    file_data = os.path.join(path_save, f"{file_stem}_traces.pkl") 
    with open(file_data, "rb") as f:
        bundle = pickle.load(f)
    traces = bundle["traces"]
    nroi = traces.shape[0]
    nframes = traces.shape[1]

    #Load peak timing data
    file_data = os.path.join(path_save, f"{file_stem}_peaks.pkl")
    with open(file_data, "rb") as f:
        bundle = pickle.load(f)
    peaks_max = bundle["peaks_max"]
    peaks_start = bundle["peaks_start"]
    
    # load filtering parameters and filter traces
    param = bundle["parameters"]["trace filtering"]
    fs =param["fs"]
    low_freq_thresh = param["low_freq_thresh"]
    high_freq_thresh = param["high_freq_thresh"]
    order = param["order"]
    traces_filt=bandpass_filter_traces(traces, fs=fs, low=low_freq_thresh, high=high_freq_thresh, order=order)

    # Initialize outputs
    t2p_fast=[]
    t2p_slow=[]
    fast_slow_ratio=[]
    # Process each ROI independently
    for roi in range(nroi):
        # Extract valid peak start and max indice
        st = peaks_start[roi][np.isfinite(peaks_start[roi])].astype(int)
        mx = peaks_max[roi][np.isfinite(peaks_max[roi])].astype(int)

        # Extract peak-aligned segments
        peaks = [] # stores segments of trace around each peak
        ln=[] # stores time-to-peak (in frames
        for i, s in enumerate(st):
            if s+fr<nframes: # ensure segment stays within bounds
                peaks.append(traces_filt[roi][s:s+fr])
                ln.append(mx[i] - s) # time-to-peak (frames)
        peaks = np.array(peaks)
        ln = np.array(ln)

        # Normalize peak shapes (z-score per peak)
        peaks_norm = zscore(peaks, axis=1)

        # Cluster peak shapes
        distance = pdist(peaks_norm, metric="euclidean") 
        Z = linkage(distance, method="ward")
        order = dendrogram(Z, no_plot=True)["leaves"]
        # Get cluster assignments (2 clusters)
        clusters = fcluster(Z, t=2, criterion="maxclust")

        # Identify which cluster corresponds to "fast" peaks
        if np.mean(ln[clusters == 1])> np.mean(ln[clusters == 2]):
                fast_clust=2
        else:
            fast_clust=1

        # Plot and save heatmap of clustered peaks
        p = plot_peak_heatmap(peaks_norm, Z, clusters, fast_clust, fr, fs)
        plt.savefig(os.path.join(folder_save, f"_roi{roi}_peak_heatmap.png"), dpi=300, bbox_inches="tight")
        plt.close()
        
        #Compute time-to-peak (frames) for all peaks
        ln = mx-st

        # Convert to seconds and separate fast/slow using threshold
        fast = ln[ln < thresh]/fs
        slow = ln[ln >= thresh]/fs

        # Store mean values (handle empty cases)
        t2p_fast.append(np.mean(fast) if fast.size > 0 else np.nan)
        t2p_slow.append(np.mean(slow) if slow.size > 0 else np.nan)

        #Compute ratio of fast to slow events
        nfast = np.sum(ln < thresh)
        nslow = np.sum(ln >= thresh)

        ratio = nfast / nslow if nslow > 0 else np.nan
        fast_slow_ratio.append(ratio)

    return t2p_fast, t2p_slow, fast_slow_ratio

   

 

In [4]:
data_source = {
"exp1_21_06_22_AL1322_P0pups_Gad-KCC2-KO":["pup2_1_spont"]
}

In [5]:
path_dist = "Data"
# Load genotype information (indexed by pup ID)
genotype=pd.read_excel(os.path.join(path_dist, "genotypes_exp1_12.xlsx"), index_col=0)

#Initialize containers for results
pup_list=[]
dt_t2p_fast=[]
dt_t2p_slow=[]
dt_fast_slow_ratio=[]
gen=[]
#Loop over experiments and recordings
for folder_exp in data_source.keys():
    pups=data_source[folder_exp]
    for file_stem in pups:
        # Create output directory for this recording
        os.makedirs(os.path.join(path_dist, "NMF_analysis_results", folder_exp, file_stem), exist_ok=True)
        path_save=os.path.join(path_dist, "NMF_analysis_results", folder_exp, file_stem)
        # Construct unified sample (pup) identifier
        pup = folder_exp.split("_")[0] + "_" + file_stem.split("_")[0]
        #Run peak-shape analysis
        t2p_fast, t2p_slow, fast_slow_ratio = main(path_save,file_stem)
        #Store aggregated metrics per sample
        pup_list.append(pup)
        dt_t2p_fast.append(np.nanmean(t2p_fast))
        dt_t2p_slow.append(np.nanmean(t2p_slow))
        dt_fast_slow_ratio.append(np.mean(fast_slow_ratio))
        print(f"Processing {file_stem} from {folder_exp}")
#Combine into a results DataFrame (one row per sample)
out=pd.DataFrame(index=pup_list, data={"t2p_fast": dt_t2p_fast, "t2p_slow": dt_t2p_slow,"fast_slow_ratio":dt_fast_slow_ratio})
#Add genotype information
out["genotype"]=genotype["genotype"]
#Save results
os.makedirs(os.path.join(path_dist,"NMF_analysis_results"), exist_ok=True)
out.to_excel(os.path.join(path_dist, "NMF_analysis_results", "nmf_compounds_t2p_statistics.xlsx"))

Processing pup2_1_spont from exp1_21_06_22_AL1322_P0pups_Gad-KCC2-KO
